## PyINE prompt result viewer

This notebook shows how to use the framework's prompt result database to display/print/plot prompting results or view them in an interactive fashion.

The code below will rely on the default prompt result database (via `pyine.prompts.get_framework_db_path()`), and it will display its content (according to some optional filters) using plots and ipywidgets.

**First, let's set up some boilerplate stuff:**

In [ ]:
import datetime
import html
import json

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display

import pyine.prompts
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()  # loads dotenv variables, seeds, sets up logging, etc.


def _escape(
    text: str,
) -> str:
    """HTML-escape and keep newlines."""
    return html.escape(text).replace("\n", "<br>")


def _collapsible_block(
    title: str,
    raw_text: str,
    *,
    threshold: int = 600,
    max_height_px: int = 420,
) -> str:
    """Return an HTML block that is collapsible if the text is long.

    Args:
        title: Section title shown in the summary.
        raw_text: Unescaped content to display inside the block.
        threshold: If len(raw_text) > threshold, the block is collapsed by default.
        max_height_px: Max visible height for the content (scrolls if larger).

    Returns:
        An HTML snippet as a string.
    """
    is_long = len(raw_text) > threshold
    escaped = _escape(raw_text)
    open_attr = "" if is_long else " open"
    summary_suffix = f" ({len(raw_text)} chars)"
    content_style = (
        "white-space:pre-wrap; border:1px solid #444; padding:8px; "
        "border-radius:8px; margin-top:8px; "
        f"max-height:{max_height_px}px; overflow:auto"
    )
    summary_text = html.escape(title)
    summary_line = f"    <summary style=\"cursor:pointer; font-weight:600\">{summary_text}{summary_suffix}</summary>"
    block_lines = [
        "<div style=\"margin-bottom:12px\">",
        f"  <details{open_attr}>",
        summary_line,
        f"    <div style=\"{content_style}\">{escaped}</div>",
        "  </details>",
        "</div>",
    ]
    return "\n".join(block_lines)


def view_llm_results(
    records: list["pyine.prompts.PromptResultRecord"],
) -> None:
    """Notebook viewer with next/prev controls for LLM results.

    Args:
        records: List of records to display.
    """
    if not records:
        display(HTML("<b>No records.</b>"))
        return

    idx_slider = widgets.IntSlider(value=0, min=0, max=len(records) - 1, step=1, description="idx")
    prev_btn = widgets.Button(description="◀ Prev")
    next_btn = widgets.Button(description="Next ▶")
    out = widgets.Output()

    def render(idx: int) -> None:
        rec = records[idx]
        small_blocks = [
            ("Identifier", f"id={rec.identifier}, group={rec.group}"),
            (
                "Prompt, version",
                f"name={rec.prompt_name}. version={rec.prompt_version}",
            ),
            ("Created at", rec.creation_meta.created_at.strftime("%Y-%m-%d %H:%M:%S")),
            ("Tags", str(rec.tags)),
        ]
        html_parts: list[str] = [
            f"""<div style="font-family: ui-monospace, SFMono-Regular, Menlo, monospace; line-height:1.35">
            <div style="margin-bottom:12px"><b>Index:</b> {idx + 1}/{len(records)}</div>
            """
        ]
        for title, text in small_blocks:
            details_html = (
                "<div style='margin-bottom:12px'>"
                f"<b>{html.escape(title)}</b>"
                "<div style='white-space:pre-wrap; border:1px solid #444; padding:8px; border-radius:8px'>"
                f"{_escape(text)}</div></div>"
            )
            html_parts.append(details_html)
        html_parts.append(_collapsible_block("Response", rec.result, threshold=800))
        html_parts.append(_collapsible_block("Prompt", rec.prompt, threshold=600))
        html_parts.append(_collapsible_block("Metadata", json.dumps(rec.meta, indent=2), threshold=400))
        html_parts.append("</div>")
        page_html = "\n".join(html_parts)
        out.clear_output(wait=True)
        with out:
            display(HTML(page_html))

    def on_prev(_: widgets.Button) -> None:
        if idx_slider.value == idx_slider.min:
            idx_slider.value = idx_slider.max
        else:
            idx_slider.value = idx_slider.value - 1

    def on_next(_: widgets.Button) -> None:
        if idx_slider.value == idx_slider.max:
            idx_slider.value = idx_slider.min
        else:
            idx_slider.value = idx_slider.value + 1

    def on_change(change: dict) -> None:
        if change["name"] == "value":
            render(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    idx_slider.observe(on_change)

    controls = widgets.HBox([prev_btn, next_btn, idx_slider])
    display(controls, out)
    render(idx_slider.value)

**Connect to the framework's default (global) prompt result database:**

In [ ]:
result_db = pyine.prompts.get_framework_db()

**Print some high-level stats about the database contents:**

In [ ]:
all_records = result_db.get_all_results()

records_data = []
for record in all_records:
    created_at = record.creation_meta.created_at.astimezone(datetime.UTC)
    tags = record.tags or []
    augment_tags = [tag for tag in tags if tag.startswith("augment:")]
    llm_params = (record.creation_meta.llm_params or {}).get("model_kwargs", {})
    llm_settings_label = json.dumps(llm_params, sort_keys=True) if llm_params else "missing"
    records_data.append(
        {
            "prompt_name": record.prompt_name,
            "prompt_name_label": record.prompt_name or "missing",
            "prompt_version": record.prompt_version,
            "prompt_version_label": record.prompt_version or "missing",
            "group": record.group,
            "group_label": record.group or "missing",
            "identifier": record.identifier,
            "created_at": created_at,
            "created_by": record.creation_meta.created_by or "missing",
            "tags": tags,
            "augment_tags": augment_tags,
            "llm_settings_label": llm_settings_label,
        },
    )
records_df = pd.DataFrame(records_data)
prompt_counts = pd.Series(dtype="int64")
group_counts = pd.Series(dtype="int64")
identifier_counts = pd.Series(dtype="int64")
augment_counts = pd.Series(dtype="int64")
created_by_counts = pd.Series(dtype="int64")
llm_counts = pd.Series(dtype="int64")
summary_findings = {}
if records_df.empty:
    print("No records in database.")
else:
    prompt_counts = records_df.groupby("prompt_name_label").size().sort_values(ascending=False)
    group_counts = records_df.groupby("group_label").size().sort_values(ascending=False)
    identifier_counts = records_df.groupby("identifier").size().sort_values(ascending=False)
    augment_counts = records_df["augment_tags"].explode().dropna().value_counts().sort_values(ascending=False)
    created_by_counts = records_df["created_by"].value_counts().sort_values(ascending=False)
    llm_counts = records_df["llm_settings_label"].value_counts().sort_values(ascending=False)
    average_stats = pd.Series(
        {
            "avg records per group": (float(group_counts.mean()) if not group_counts.empty else 0.0),
            "avg records per identifier": (float(identifier_counts.mean()) if not identifier_counts.empty else 0.0),
        },
        name="value",
    )
    display(average_stats.to_frame())
    display(prompt_counts.rename("records").to_frame())
    display(group_counts.rename("records").to_frame())
    display(identifier_counts.rename("records").to_frame())
    display(augment_counts.rename("records").to_frame())
    if not prompt_counts.empty:
        summary_findings["prompt_name"] = (
            prompt_counts.index[0],
            int(prompt_counts.iloc[0]),
        )
    if not group_counts.empty:
        summary_findings["group"] = (group_counts.index[0], int(group_counts.iloc[0]))
    if not identifier_counts.empty:
        summary_findings["identifier"] = (
            identifier_counts.index[0],
            int(identifier_counts.iloc[0]),
        )
    if not augment_counts.empty:
        summary_findings["augment_tag"] = (
            augment_counts.index[0],
            int(augment_counts.iloc[0]),
        )
    if not created_by_counts.empty:
        summary_findings["created_by"] = (
            created_by_counts.index[0],
            int(created_by_counts.iloc[0]),
        )
    if not llm_counts.empty:
        summary_findings["llm_settings"] = (
            llm_counts.index[0],
            int(llm_counts.iloc[0]),
        )
    prompt_name_counts_actual = records_df["prompt_name"].dropna().value_counts()
    if not prompt_name_counts_actual.empty:
        top_prompt_name_value = prompt_name_counts_actual.index[0]
        summary_findings["prompt_name_value"] = (
            top_prompt_name_value,
            int(prompt_name_counts_actual.iloc[0]),
        )
        version_counts = (
            records_df.loc[records_df["prompt_name"] == top_prompt_name_value, "prompt_version"].dropna().value_counts()
        )
        if not version_counts.empty:
            summary_findings["prompt_version_value"] = (
                version_counts.index[0],
                int(version_counts.iloc[0]),
            )

**Display histograms describing the contents of the database:**

In [ ]:
if records_df.empty:
    print("No records available for visualization.")
else:
    data_frame = records_df.copy()
    now_utc = datetime.datetime.now(datetime.UTC)
    data_frame["age_days"] = (now_utc - data_frame["created_at"]).dt.total_seconds() / 86400.0
    augment_counts = (
        data_frame["augment_tags"]
        .explode()
        .dropna()
        .value_counts()
        .rename_axis("augment_tag")
        .reset_index(name="count")
    )
    created_by_counts = data_frame["created_by"].value_counts().rename_axis("created_by").reset_index(name="count")
    llm_counts = data_frame["llm_settings_label"].value_counts().rename_axis("llm_settings").reset_index(name="count")
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    sns.histplot(data_frame["age_days"], bins="auto", ax=axes[0], color="#4c72b0")
    axes[0].set_title("Result age (days)")
    axes[0].set_xlabel("days since creation")
    axes[0].set_ylabel("records")
    if augment_counts.empty:
        axes[1].text(0.5, 0.5, "No augment:* tags", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        sns.barplot(data=augment_counts, x="count", y="augment_tag", ax=axes[1], color="#dd8452")
        axes[1].set_title("augment:* tags")
        axes[1].set_xlabel("records")
        axes[1].set_ylabel("tag")
    if created_by_counts.empty:
        axes[2].text(0.5, 0.5, "No created_by metadata", ha="center", va="center")
        axes[2].set_axis_off()
    else:
        sns.barplot(
            data=created_by_counts,
            x="count",
            y="created_by",
            ax=axes[2],
            color="#55a868",
        )
        axes[2].set_title("Records by created_by")
        axes[2].set_xlabel("records")
        axes[2].set_ylabel("user")
    if llm_counts.empty:
        axes[3].text(0.5, 0.5, "No LLM settings metadata", ha="center", va="center")
        axes[3].set_axis_off()
    else:
        sns.barplot(data=llm_counts, x="count", y="llm_settings", ax=axes[3], color="#c44e52")
        axes[3].set_title("Records by LLM settings")
        axes[3].set_xlabel("records")
        axes[3].set_ylabel("settings")
    for axis in axes.flatten():
        if axis.axison:
            axis.grid(axis="x", linestyle="--", alpha=0.3)
    fig.tight_layout()
    plt.show()

## Interactive database parsing and visualization

In [ ]:
# for the interactive visualization: target a specific prompt name, and display all results for it

target_prompt_name = "hints/docs"  # MODIFY ME IF NEEDED!
target_prompt_version = None  # MODIFY ME IF NEEDED!
tag_filter_rule = None  # MODIFY ME IF NEEDED!
max_result_age = None  # MODIFY ME IF NEEDED!

if target_prompt_name not in result_db.list_prompt_names():
    raise ValueError(f"prompt name '{target_prompt_name}' not found in database")

found_records = result_db.get_by_prompt_name(
    prompt_name=target_prompt_name,
    prompt_version=target_prompt_version,
    tag_filter_rule=tag_filter_rule,
    max_result_age=max_result_age,
)

view_llm_results(found_records)